# Chapter 6 — Assay Curation (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

**Learning objectives**
- Normalize units and relations (<, >, =) in assay data
- Record endpoint context and transformations
- Detect duplicate/conflicting measurements
- Export a curated, documented table

> Runtime: ~5 min (local, no API)  
> Cost: free  
> Data: small synthetic assay set

> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.


## Environment setup


### Secrets (optional LLM only)


In [1]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

✅ API keys loaded for OPENAI (source: Colab Secrets)


### Install pinned dependencies


In [2]:
# @title Installing Python dependencies
%pip install -q "rdkit==2023.9.6" "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "pandas>=2.0" "matplotlib>=3.8" "scipy>=1.11" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/34.8 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.7 MB/s eta 0:00:00


In [3]:
# @title Setting LangSmith variables
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter6-assay-curation"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF (fine — these notebooks are local/RDKit-first)")

LangSmith OFF (fine — these notebooks are local/RDKit-first)


## Why curate assays?

Raw bioactivity data mixes units (nM/µM), relations (`<`, `>`, `=`), and endpoints (IC50/Ki/EC50). Modeling on uncurated data silently mixes apples and oranges. Curation makes every value **explicit and traceable**.


## 1. Load a raw assay table


In [4]:
import pandas as pd

raw = pd.DataFrame(
    {
        "compound": ["A", "B", "C", "D", "E", "F"],
        "endpoint": ["IC50", "IC50", "Ki", "IC50", "EC50", "IC50"],
        "relation": ["=", "<", "=", ">", "=", "="],
        "value": [10.0, 5.0, 200.0, 1000.0, 50.0, 10.0],
        "unit": ["nM", "nM", "nM", "nM", "uM", "nM"],
    }
)
print(raw)

  compound endpoint relation   value unit
0        A     IC50        =    10.0   nM
1        B     IC50        <     5.0   nM
2        C       Ki        =   200.0   nM
3        D     IC50        >  1000.0   nM
4        E     EC50        =    50.0   uM
5        F     IC50        =    10.0   nM


## 2. Normalize units to a single scale (nM)


In [5]:
UNIT_TO_NM = {"nM": 1.0, "uM": 1000.0, "µM": 1000.0, "pM": 0.001, "mM": 1e6}

raw["value_nM"] = raw.apply(lambda r: r["value"] * UNIT_TO_NM[r["unit"]], axis=1)
raw["transform"] = raw.apply(
    lambda r: f'{r["value"]}{r["unit"]} -> {r["value_nM"]}nM', axis=1
)
print(raw[["compound", "transform"]])

  compound             transform
0        A      10.0nM -> 10.0nM
1        B        5.0nM -> 5.0nM
2        C    200.0nM -> 200.0nM
3        D  1000.0nM -> 1000.0nM
4        E   50.0uM -> 50000.0nM
5        F      10.0nM -> 10.0nM


## 3. Make relations explicit (censoring)


In [6]:
def censor(row):
    if row["relation"] == "<":
        return f'upper_bound={row["value_nM"]}nM (true value below)'
    if row["relation"] == ">":
        return f'lower_bound={row["value_nM"]}nM (true value above)'
    return f'exact={row["value_nM"]}nM'


raw["interpretation"] = raw.apply(censor, axis=1)
print(raw[["compound", "relation", "interpretation"]])

  compound relation                           interpretation
0        A        =                             exact=10.0nM
1        B        <     upper_bound=5.0nM (true value below)
2        C        =                            exact=200.0nM
3        D        >  lower_bound=1000.0nM (true value above)
4        E        =                          exact=50000.0nM
5        F        =                             exact=10.0nM


## 4. Endpoint context & comparability


In [7]:
ENDPOINT_KIND = {
    "IC50": "functional inhibition",
    "Ki": "binding affinity",
    "EC50": "functional activation",
    "IC90": "functional inhibition (90%)",
}
raw["endpoint_kind"] = raw["endpoint"].map(ENDPOINT_KIND)
print(raw[["endpoint", "endpoint_kind"]].drop_duplicates())
print("NOTE: do not pool Ki (binding) with IC50 (functional) without justification.")

  endpoint          endpoint_kind
0     IC50  functional inhibition
2       Ki       binding affinity
4     EC50  functional activation
NOTE: do not pool Ki (binding) with IC50 (functional) without justification.


## 5. Duplicate / conflicting measurements


In [8]:
dup = raw[raw.duplicated(["compound", "endpoint"], keep=False)]
conflicts = dup.groupby(["compound", "endpoint"])["value_nM"].agg(
    ["min", "max", "count"]
)
conflicts["fold_range"] = conflicts["max"] / conflicts["min"]
print("Repeated compound+endpoint measurements:")
print(conflicts)

Repeated compound+endpoint measurements:
Empty DataFrame
Columns: [min, max, count, fold_range]
Index: []


## 6. Export curated table


In [9]:
curated = raw[
    [
        "compound",
        "endpoint",
        "endpoint_kind",
        "relation",
        "value_nM",
        "interpretation",
        "transform",
    ]
]
curated.to_csv("curated_assay.csv", index=False)
print(curated.to_string(index=False))
print("Wrote curated_assay.csv")

compound endpoint         endpoint_kind relation  value_nM                          interpretation            transform
       A     IC50 functional inhibition        =      10.0                            exact=10.0nM     10.0nM -> 10.0nM
       B     IC50 functional inhibition        <       5.0    upper_bound=5.0nM (true value below)       5.0nM -> 5.0nM
       C       Ki      binding affinity        =     200.0                           exact=200.0nM   200.0nM -> 200.0nM
       D     IC50 functional inhibition        >    1000.0 lower_bound=1000.0nM (true value above) 1000.0nM -> 1000.0nM
       E     EC50 functional activation        =   50000.0                         exact=50000.0nM  50.0uM -> 50000.0nM
       F     IC50 functional inhibition        =      10.0                            exact=10.0nM     10.0nM -> 10.0nM
Wrote curated_assay.csv


## Limitations & safety notes

- Unit conversion assumes the `UNIT_TO_NM` map; extend/validate for your data.
- `<`/`>` values are censored, not exact.
- Pooling different endpoints (Ki vs IC50) needs explicit scientific justification.
- Local/free; no API needed.


In [10]:
# Cleanup
import gc

for _v in ("mol", "mols", "df", "llm", "model", "img", "raw", "curated"):
    globals().pop(_v, None)
try:
    import torch

    torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()
print("Cleanup complete.")

Cleanup complete.


## Exercises

<details><summary>Why keep the relation (<, >, =)?</summary>It marks censored data; dropping it turns bounds into fake exact values and biases models.</details>

<details><summary>Why not pool Ki and IC50?</summary>They measure different things (binding vs functional response); combining them adds noise/error.</details>

<details><summary>Why log transformations?</summary>So every normalized value is traceable back to its raw value and unit.</details>

### Tasks
- **Task A** - Add a `pchembl` (-log10 molar) column computed from `value_nM`.
- **Task B** - Flag measurements with fold_range > 3 as conflicts needing review.
- **Task C** - Add a units-validation step that errors on an unknown unit string.
- **Task D** - Export a JSON manifest recording unit map, relation policy, and row counts.
